In [1]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_READ_DATASETS_TOKEN")

# Authenticate to Hugging Face
login(token=hf_token)

In [2]:
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -U bitsandbytes>=0.46.1

import accelerate
import transformers
import torch
print("accelerate:", accelerate.__version__)   # should be 1.x+
print("transformers:", transformers.__version__)

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-4t_1b4ze
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-4t_1b4ze
  Resolved https://github.com/huggingface/transformers.git to commit 31cd7301503b14b1d002aaab34d24909c37f261a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.10.0.dev0-py3-none-any.whl size=12039701 sha256=95d0521e810885c1509397283c5ebd81840ef5786bdfff55768d8eac69a73515
  Stored in directory: /tmp/pip-ephem-wheel-cache-ous37mtf/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolv

In [3]:
MEDGEMMA_1_5_4B_IT_ID = "google/medgemma-1.5-4b-it"
MEDGEMMA_27B_TEXT_ID = "google/medgemma-27b-text-it"

# should the model think to reason over the clinical signals? Yes.
is_thinking = True # set the boolean flag for thinking

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# configure for 4-bit memory quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MEDGEMMA_1_5_4B_IT_ID)
model = AutoModelForCausalLM.from_pretrained(
    MEDGEMMA_1_5_4B_IT_ID,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",
    attn_implementation="sdpa" # for long thinking and reasoning
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [5]:
model.device

device(type='cuda', index=0)

```text
Input: Agent 1 clinical profile JSON
          │
          ▼
    extraction_confidence == "low"?
          │
    ┌─────┴─────┐
   Yes          No
    │            │
    ▼            ▼
 Pass 1:     Single call:
 Reason      Reason over
 normally    signals →
    │        structured
    ▼        output
 Pass 2:
 Self-critique
 under uncertainty
    │
    ▼
 Revised output
 with confidence
 flags per pattern
          │
          ▼
    No abnormal findings?
          │
    ┌─────┴─────┐
   Yes          No
    │            │
    ▼            ▼
Constructive   Full pattern
description    analysis
of normal      output
state
          │
          ▼
    Output: signal analysis JSON → Agent 3

In [6]:
single_call_system_prompt = """
You are an expert Medical Practitioner who is skilled in interpreting data directly from the clinical signals that are provided to make intermediate medical decisions that are informative to the patient about their health, to inform them about what they may be looking at in their health report, lab tests, or any other medical diagnostic document/report that they might have been handed.

You will receive a clinical profile JSON containing: "document_type", "patient_context" (age, sex, history), "lab_findings" (each with the name, value, unit, range, and status), imaging_findings, clinical_notes, and flagged_signals.

The clinical signals provided to you are elevated markers, abnormal findings, flagged terms, and other details that have been extracted from the documents provided by the patient by another medical document analyst who is skilled in extracting such clinical signals. What you have with you is a full clinical profile of the patient. 

Reason over this clinical profile, performing rigorous cross-signal reasoning, and identify what is unusual, namely:
    - Identify and understand what signals appear together meaningfully and why
    - What this combination suggests or is trying to show is that either signal individually cannot
    - How frequently this combination occurs and the cases in which it occurs
    - The kind of scenarios, circumstances, and cases in which this combination occurs
    - The criticality of this signal and whether it is something urgent that the patient should be consulting their doctor/clinician immediately
    - Is the combination something that will progressively develop, or is it static (mention this in brief for the patient to understand)

Post your reasoning for the requirements needed to thoroughly understand the interplay and co-occurrence of multiple signals, you need to extract them carefully with accuracy and precision. Construct a structured signal analysis object that includes all the details that you identified through your cross-signal reasoning, and provide the signal analysis object as a single JSON object, following the provided JSON schema below strictly:

{
  "individual_signals": [
    {
        "signal": "string - copied exactly from flagged_signals in the input",
        "reason": "string - copied exactly from flagged_signals in the input",
    }
  ],
  "combination_patterns": [
    {
      "signals_involved": ["signal name 1", "signal name 2", "..."],
      "pattern_description": "string — what this combination suggests clinically for each tuple in signals_involved, and their overall implication",
      "co-occurrence_context": "string - how commonly these clinical signals appear together and in what clinical scenarios",
      "clinical_significance": "high | medium | low",
      "is_progressive": "true | false",
      "suggested_investigation": "string — what kind of specialist or test this points to"
    }
  ],
  "overall_assessment": "string — plain language summary of what the patient should understand",
  "urgency": "routine | soon | urgent",
  "analysis_confidence": "high | medium | low"
}

If you do not find any abnormal findings, do not force yourself to find or invent new combination patterns that do not exist. Set combination_patterns to an empty array, present the findings constructively, as to your reasoning behind why there are no abnormal findings, and use overall_assessment to describe what the normal findings collectively indicate about the patient's health.

Return ONLY the JSON object. No preamble, introductory notes, explanation, or markdown code fences. The first character of your response should be the opening braces { of the JSON object, and the last character of your response should be the closing braces } of the JSON object.
"""